THEORY: `realloc()` - How, When, and Why
   WHY: Arrays in C are strictly fixed in size. You cannot append to a full
      memory block natively.
   WHEN: You use `realloc()` when the dynamic structure (like `IntVec`) reaches
      its capacity, and you need a larger contiguous block of memory without
      losing your existing data.
   HOW: `void *realloc(void *ptr, size_t new_size);`

   1. It looks at the existing heap block pointed to by `ptr`.
   2. It asks the OS for a new block of `new_size` bytes.
   3. If there is enough contiguous free space immediately after the current 
      block, it just extends the boundary (fast).
   4. If not, it allocates a brand new block... COPIES old data byte-for-byte
      ... frees old pointer, return new pointer (slow)...

```c
arr = realloc(arr, 20 * sizeof *arr);
```

---

```c
v->data = realloc(v->data, new_cap * sizeof *v->data);
```
   When `realloc` fails, `v->data` will now point to `NULL` instead. This means
   that the original memory block's data is just lost forever, and isn't freed
   as well, which means that the allocated memory will be held forever, which is
   a memory leak, as it will eventually pile up. And since we have also lost
   reference which points to the original memory block, this means that we won't
   be able to call `free()` later even if desired. Hence, a memory leak. 
        // If `realloc` fails, it returns `NULL`. Because you assigned the 
        // result directly back to `v->data`, `v->data` is now `NULL`.
        -
        // However, `realloc` didn't free the original memory block. You have
        // now completely lost the memory address of your original array, 
        // creating an unrecoverable memory leak.

SAFE PATTERN
```c
int *temp = realloc(v->data, new_cap * sizeof *v->data);
if (temp == NULL) {
    return INTVEC_ERR_ALLOC;        // Original v->data is still perfectly safe and valid.
}
v->data = temp;
```

---

```c
size_t bytes = 128;
malloc(bytes);          // Equiv to `realloc(NULL, bytes);`
```
   This is incedibly useful because it means you do not need to write separate
   logic in your `intvec_push` function for "first allocation" versus 
   "subsequent resizing."

   If `v->data` is initialized to `NULL` (e.g., in `intvec_init`), you can just 
   safely call `realloc(v->data, new_bytes)` during your very first push, and
   it will seamlessly act as a `malloc`.

   (Note: The reverse is historically true too--`realloc(ptr, 0)` traditionally
   behaves like `free(ptr)`, though modern C standards consider this specific
   edge cas implementation-defined or deprecated. Rely on the `NULL` trick, avoid
   the `0` trick).

---

```c
int intvec_push(IntVec *v, int value) {
    if (v->len == v->cap) {
        size_t new_cap = (v->cap == 0) ? 16 : v->cap * 2;
        int *temp = realloc(v->data, new_cap * sizeof *v->data);
        if (temp == NULL) { return INTVEC_ERROR_ALLOC; }
        v->data = temp;
        v->cap = new_cap;
    }
    v->data[v->len] = value;
    v->len++;
    return INTVEC_OK;
}
```

---

   The `realloc()` function in the C programming language is mainly used for
   deallocating the old memory of the variable pointed to by the pointer and then
   returns a pointer to a new variable that has the size specified by `size`.

   

   ... `sizeof *v->data` ... paradox: HOW CAN WE EVALUATE THE SIZE OF WHAT A 
   POINTER POINTS TO IF THE POINTER IS CURRENTLY UNINITIALISED OR EMPTY?

THE SECRET: `sizeof` is a compile-time operator
   The magic here is that `sizeof` NEVER ACTUALLY EXECUTES CODE AT RUNTIME. It
   does not look at the memory address stored inside `v->data`, nor does it 
   attempt to dereference it to look at a value. 

   Instead, the compiler does a pure static type-check.
   1. ... compiler looks at the header file and sees: `int *data`.
   2. It reasons: "The type of `v->data` is a pointer to an `int`"
   3. Therefirem the type of the expression `*v->Data` must be a plain `int`
   4. The compiler completely replaces the text `sizeof *v->data` with the 
      literal number `4`... before generating the binary.

      

   ... misunderstanding which pointer is `NULL`.

   The error is not complaining that `v->data` is `NULL`.
   The error is complaining that `v` itself is `NULL`.

   The test suite is actively trying to break your API by calling 
   `intvec_init(NULL);`.

   When your code hits `v->data = malloc(...);`, C tries to find the memory
   address of `v` so it can access the `data` field. But because `v` is
   `0x000000000000`, the operating system immediately terminates the program.

   ...

   In C, you can never trust the caller. If a function takes a pointer as an
   argument, teh very first thing you must do is prove that pointer actually
   exists.

   ... need to add a "`NULL` guard" at the very top of `intvec_init`...